In [ ]:
# load the API Key
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from openai import OpenAI

# loading api key
load_dotenv("../.env")
openai_api_key = os.getenv("OPENAI_API_KEY")

# apply your api key
client = OpenAI(api_key=openai_api_key)
# open file 
file = open("meeting_note.txt", "r", encoding="utf-8")
query = '\n'.join(file.readlines())

In [16]:
def generate(query):
    messages = [
        {"role": "system", "content": "다음은 마케팅 부서 회의의 미팅 노트입니다. 이 노트를 바탕으로 정돈된 보고서를 작성하세요. "},
        {"role": "system", "content": "보고서에는 회의의 주요 논의 사항, 결론, 후속 조치 계획이 포함되어야 합니다. "},
        {"role": "system", "content": "미팅 노트는 다음과 같습니다: {query}"}
    ]
    response = client.chat.completions.create(
        model="gpt-5.6-luna",
        messages=messages,
        n=1,
        max_completion_tokens=1500)
    
    queries = response.choices[0].message.content.split('\n')
    queries = [q for q in queries if any(c.isalpha() for c in q)]
    return queries


In [18]:
# result = generate(query)
# print("\n".join(result))

In [20]:
# response = client.responses.create(
#     model="gpt-5.5",
#     instructions="You are a coding assistant that talks like a pirate.",
#     input="How do I check if a Python object is an instance of a class?",
# )

# print(response.output_text)

In [23]:
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"

 # "gpt2-xl" for larger version
model_name = "gpt2"
tokenizer  = AutoTokenizer.from_pretrained(model_name)
model      = AutoModelForCausalLM.from_pretrained(model_name).to(device)

input_txt  = "Transformers are the"
input_ids  = tokenizer(input_txt, return_tensors="pt")["input_ids"].to(device)

c:\Users\jia\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\jia\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jia\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to

Transformers are the most common type of particle. They are


In [25]:
n_steps    = 10

output = model.generate(input_ids, max_new_tokens=n_steps, do_sample=False)
print(tokenizer.decode(output[0]))

Transformers are the most common type of particle. They are composed of


In [29]:
max_length = 128
input_txt = """In a shocking finding, scientist discovered \
a herd of unicorns living in a remote, previously unexplored \
valley, in the Andes Mountains. Even more surprising to the \
researchers was the fact that the unicorns spoke perfect English.\n\n
"""
input_ids = tokenizer(input_txt, return_tensors="pt")["input_ids"].to(device)
output_greedy = model.generate(input_ids, max_length=max_length, do_sample=False)
output_beam = model.generate(input_ids, max_length=max_length,num_beams=5, do_sample=False)
print(tokenizer.decode(output_greedy[0]))
print("Beam output>>>>>>>> \n",tokenizer.decode(output_beam[0]))

In a shocking finding, scientist discovered a herd of unicorns living in a remote, previously unexplored valley, in the Andes Mountains. Even more surprising to the researchers was the fact that the unicorns spoke perfect English.


"The unicorns were very intelligent, and they were very intelligent," said Dr. David S. Siegel, a professor of anthropology at the University of California, Berkeley. "They were very intelligent, and they were very intelligent, and they were very intelligent, and they were very intelligent, and they were very intelligent, and they were very intelligent, and they were very intelligent, and they were very
Beam output>>>>>>>> 
 In a shocking finding, scientist discovered a herd of unicorns living in a remote, previously unexplored valley, in the Andes Mountains. Even more surprising to the researchers was the fact that the unicorns spoke perfect English.


The researchers, from the University of California, San Diego, and the University of California, Santa Cr

#### Direct implementation

In [26]:
n_steps = 8
choices_per_step = 5
iterations = []

with torch.no_grad():
    for _ in range(n_steps):
        iteration = dict()
        iteration["Input"] = tokenizer.decode(input_ids[0])
        output = model(input_ids=input_ids)

        # Select logits of the first batch and the last token and apply softmax
        next_token_logits = output.logits[0, -1, :]
        next_token_probs = torch.softmax(next_token_logits, dim=-1)
        sorted_ids = torch.argsort(next_token_probs, dim=-1, descending=True)

        # Store tokens with highest probabilities
        for choice_idx in range(choices_per_step):
            token_id = sorted_ids[choice_idx]
            token_prob = next_token_probs[token_id].cpu().numpy()
            token_choice = f"{tokenizer.decode(token_id)} ({100 * token_prob:.2f}%)"
            iteration[f"Choice {choice_idx+1}"] = token_choice
            
            # Append predicted next token to input
        input_ids = torch.cat([input_ids, sorted_ids[None, 0, None]], dim=-1)
        iterations.append(iteration)
        
pd.set_option('display.max_columns', None) # Print all columns
pd.set_option('display.expand_frame_repr', False) # Print without scrolling sideways
print(pd.DataFrame(iterations))

                                               Input           Choice 1           Choice 2            Choice 3          Choice 4              Choice 5
0                               Transformers are the       most (9.76%)       same (2.94%)        only (2.87%)      best (2.38%)         first (1.77%)
1                          Transformers are the most    common (22.90%)   powerful (6.88%)   important (6.32%)   popular (3.95%)      commonly (2.14%)
2                   Transformers are the most common      type (15.06%)      types (3.31%)        form (1.91%)       way (1.89%)           and (1.49%)
3              Transformers are the most common type        of (83.13%)         in (3.16%)           . (1.92%)         , (1.63%)           for (0.88%)
4           Transformers are the most common type of   particle (1.55%)     object (1.02%)       light (0.71%)    energy (0.67%)       objects (0.66%)
5  Transformers are the most common type of particle         . (14.26%)        in (11.57%)    